# 效能調校(Fine Tuning)作法，修改自
https://colab.research.google.com/github/huggingface/notebooks/blob/master/examples/text_classification.ipynb

In [1]:
! pip install datasets 

In [2]:
import random
from typing import Any, Dict, cast

import datasets
import evaluate
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import BestRun, EvalPrediction

## 參數設定

## 定義GLUE所有任務(Task)

In [3]:
GLUE_TASKS = ["cola", "mnli", "mnli-mm", "mrpc", "qnli", "qqp", "rte", "sst2", "stsb", "wnli"]

## 指定任務為 cola

In [4]:
task = "cola"
# 預先訓練模型
model_checkpoint = "distilbert-base-uncased"
# 批量
batch_size = 16

## 載入資料集、效能衡量指標

In [5]:
actual_task = "mnli" if task == "mnli-mm" else task
# 載入資料集
dataset = datasets.load_dataset("nyu-mll/glue", actual_task)
# 載入效能衡量指標
metric = evaluate.load('glue', actual_task)

### dataset 資料型態為 [`DatasetDict`](https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasetdict)

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})

## 顯示第一筆內容

In [7]:
dataset["train"][0]

{'sentence': "Our friends won't buy this analysis, let alone the next one we propose.",
 'label': 1,
 'idx': 0}

## 定義隨機抽取數據函數

In [8]:
# 隨機抽取資料函數
def show_random_elements(dataset: datasets.Dataset, num_examples: int = 10) -> None:
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset) - 1)
        while pick in picks:
            pick = random.randint(0, len(dataset) - 1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

## 查看前30筆資料

In [9]:
df = pd.DataFrame(dataset["train"][:30])
df

,sentence,label,idx
0,"Our friends won't buy this analysis, let alone...",1,0
1,One more pseudo generalization and I'm giving up.,1,1
2,One more pseudo generalization or I'm giving up.,1,2
3,"The more we study verbs, the crazier they get.",1,3
4,Day by day the facts are getting murkier.,1,4
5,I'll fix you a drink.,1,5
6,Fred watered the plants flat.,1,6
7,Bill coughed his way out of the restaurant.,1,7
8,We're dancing the night away.,1,8
9,Herman hammered the metal flat.,1,9


## 隨機抽取10筆資料查看

In [10]:
show_random_elements(dataset["train"])

,sentence,label,idx
0,Fred talked about everything before Rusty did.,acceptable,950
1,John made a fool of himself in front of anyone who was there.,acceptable,6479
2,I have sent 0 letter to Environmental Heath,unacceptable,8430
3,Which attorney will give the closing argument hasn't been decided by them.,acceptable,4671
4,I was eating not a peach but an apple,acceptable,7876
5,John said Mary went to the store quickly.,acceptable,5730
6,John believes himself to be the best at baseball.,acceptable,6342
7,John to call would be unlikely.,unacceptable,413
8,The sea monster drowned the sailors.,acceptable,3157
9,The boy swims.,acceptable,4148


The metric is an instance of [`datasets.Metric`](https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasets.Metric):

## 顯示效能衡量指標

In [11]:
metric

EvaluationModule(name: "glue", module_type: "metric", features: {'predictions': Value('int64'), 'references': Value('int64')}, usage: """
Compute GLUE evaluation metric associated to each GLUE dataset.
Args:
    predictions: list of predictions to score.
        Each translation should be tokenized into a list of tokens.
    references: list of lists of references for each translation.
        Each reference should be tokenized into a list of tokens.
Returns: depending on the GLUE subset, one or several of:
    "accuracy": Accuracy
    "f1": F1 score
    "pearson": Pearson Correlation
    "spearmanr": Spearman Correlation
    "matthews_correlation": Matthew Correlation
Examples:

    >>> glue_metric = evaluate.load('glue', 'sst2')  # 'sst2' or any of ["mnli", "mnli_mismatched", "mnli_matched", "qnli", "rte", "wnli", "hans"]
    >>> references = [0, 1]
    >>> predictions = [0, 1]
    >>> results = glue_metric.compute(predictions=predictions, references=references)
    >>> print(results

## 產生兩筆隨機亂數，測試效能衡量指標

In [12]:
fake_preds = np.random.randint(0, 2, size=(64,))
fake_labels = np.random.randint(0, 2, size=(64,))
metric.compute(predictions=fake_preds, references=fake_labels)

{'matthews_correlation': 0.11559657831495045}

Note that `load_metric` has loaded the proper metric associated to your task, which is:

- for CoLA: [Matthews Correlation Coefficient](https://en.wikipedia.org/wiki/Matthews_correlation_coefficient)
- for MNLI (matched or mismatched): Accuracy
- for MRPC: Accuracy and [F1 score](https://en.wikipedia.org/wiki/F1_score)
- for QNLI: Accuracy
- for QQP: Accuracy and [F1 score](https://en.wikipedia.org/wiki/F1_score)
- for RTE: Accuracy
- for SST-2: Accuracy
- for STS-B: [Pearson Correlation Coefficient](https://en.wikipedia.org/wiki/Pearson_correlation_coefficient) and [Spearman's_Rank_Correlation_Coefficient](https://en.wikipedia.org/wiki/Spearman%27s_rank_correlation_coefficient)
- for WNLI: Accuracy

so the metric object only computes the one(s) needed for your task.

## 分詞

In [13]:
# 分詞
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

You can directly call this tokenizer on one sentence or a pair of sentences:

## 測試兩筆資料，進行分詞

In [14]:
tokenizer("Hello, this one sentence!", "And this sentence goes with it.")

{'input_ids': [101, 7592, 1010, 2023, 2028, 6251, 999, 102, 1998, 2023, 6251, 3632, 2007, 2009, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

## 定義任務的資料集欄位

In [15]:
task_to_keys = {
    "cola": ("sentence", None),
    "mnli": ("premise", "hypothesis"),
    "mnli-mm": ("premise", "hypothesis"),
    "mrpc": ("sentence1", "sentence2"),
    "qnli": ("question", "sentence"),
    "qqp": ("question1", "question2"),
    "rte": ("sentence1", "sentence2"),
    "sst2": ("sentence", None),
    "stsb": ("sentence1", "sentence2"),
    "wnli": ("sentence1", "sentence2"),
}

## 測試第一筆資料

In [16]:
sentence1_key, sentence2_key = task_to_keys[task]
if sentence2_key is None:
    print(f"Sentence: {dataset['train'][0][sentence1_key]}")
else:
    print(f"Sentence 1: {dataset['train'][0][sentence1_key]}")
    print(f"Sentence 2: {dataset['train'][0][sentence2_key]}")

Sentence: Our friends won't buy this analysis, let alone the next one we propose.


## 測試 5 筆資料分詞

In [17]:
def preprocess_function(examples: Dict[str, Any]) -> Dict[str, Any]:
    if sentence2_key is None:
        return tokenizer(examples[sentence1_key], truncation=True)
    return tokenizer(examples[sentence1_key], examples[sentence2_key], truncation=True)


preprocess_function(dataset['train'][:5])

{'input_ids': [[101, 2256, 2814, 2180, 1005, 1056, 4965, 2023, 4106, 1010, 2292, 2894, 1996, 2279, 2028, 2057, 16599, 1012, 102], [101, 2028, 2062, 18404, 2236, 3989, 1998, 1045, 1005, 1049, 3228, 2039, 1012, 102], [101, 2028, 2062, 18404, 2236, 3989, 2030, 1045, 1005, 1049, 3228, 2039, 1012, 102], [101, 1996, 2062, 2057, 2817, 16025, 1010, 1996, 13675, 16103, 2121, 2027, 2131, 1012, 102], [101, 2154, 2011, 2154, 1996, 8866, 2024, 2893, 14163, 8024, 3771, 1012, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

To apply this function on all the sentences (or pairs of sentences) in our dataset, we just use the `map` method of our `dataset` object we created earlier. This will apply the function on all the elements of all the splits in `dataset`, so our training, validation and testing data will be preprocessed in one single command.

In [18]:
# 將所有資料進行分詞
encoded_dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

## 效能微調(Fine tuning)，先加載預先訓練的模型

In [19]:
# 載入預先訓練的模型
num_labels = 3 if task.startswith("mnli") else 1 if task == "stsb" else 2
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 定義訓練參數，可參閱 [`TrainingArguments`](https://huggingface.co/transformers/main_classes/trainer.html#transformers.TrainingArguments)

In [20]:
metric_name = "pearson" if task == "stsb" else "matthews_correlation" if task == "cola" else "accuracy"

args = TrainingArguments(
    "test-glue",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
)

## 定義效能衡量指標計算的函數

In [21]:
def compute_metrics(eval_pred: EvalPrediction) -> Dict[str, float]:
    predictions, labels = eval_pred
    predictions = cast(np.ndarray, predictions)
    if task != "stsb":
        predictions = np.argmax(predictions, axis=1)
    else:
        predictions = predictions[:, 0]
    result = metric.compute(predictions=predictions, references=labels)
    assert result is not None
    return result

## 定義訓練者(Trainer)物件

In [22]:
validation_key = (
    "validation_mismatched" if task == "mnli-mm" else "validation_matched" if task == "mnli" else "validation"
)

trainer = Trainer(
    model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset[validation_key],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

## 模型訓練

In [23]:
trainer.train()

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.521346,0.457426,0.455341
2,0.339738,0.463496,0.520013
3,0.237527,0.600112,0.536040
4,0.171659,0.803512,0.535135
5,0.126709,0.818805,0.550079


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2675, training_loss=0.2696611764497846, metrics={'train_runtime': 541.7671, 'train_samples_per_second': 78.918, 'train_steps_per_second': 4.938, 'total_flos': 229000686898068.0, 'train_loss': 0.2696611764497846, 'epoch': 5.0})

## 模型評估

In [24]:
trainer.evaluate()

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Matthews Correlation
0.126709,0.818805,5,0.550079


{'eval_loss': 0.8188051581382751,
 'eval_matthews_correlation': 0.5500793243102087}

## 模型存檔

In [25]:
trainer.save_model('cola')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 預測

In [26]:
class SimpleDataset(Dataset):
    def __init__(self, tokenized_texts: Any) -> None:
        self.tokenized_texts = tokenized_texts

    def __len__(self) -> int:
        return len(self.tokenized_texts["input_ids"])

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return {k: v[idx] for k, v in self.tokenized_texts.items()}


texts = ["Hello, this one sentence!", "And this sentence goes with it."]
tokenized_texts = tokenizer(texts, padding=True, truncation=True)
new_dataset = SimpleDataset(tokenized_texts)
trainer.predict(new_dataset)

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


PredictionOutput(predictions=array([[-1.3833233,  1.2554325],
       [-2.8858545,  2.791675 ]], dtype=float32), label_ids=None, metrics={'test_runtime': 0.1675, 'test_samples_per_second': 11.94, 'test_steps_per_second': 5.97})

In [27]:
tokenized_texts = tokenizer(
    ["They drank the pub.", "The professor talked us into a stupor."], padding=True, truncation=True
)
new_dataset = SimpleDataset(tokenized_texts)
trainer.predict(new_dataset)

PredictionOutput(predictions=array([[-2.8101385,  2.7338908],
       [-2.9691381,  2.8696914]], dtype=float32), label_ids=None, metrics={'test_runtime': 0.025, 'test_samples_per_second': 79.958, 'test_steps_per_second': 39.979})

In [28]:
tokenized_texts = tokenizer(["Hello there!", "This is another text"], padding=True, truncation=True)
new_dataset = SimpleDataset(tokenized_texts)
trainer.predict(new_dataset)

PredictionOutput(predictions=array([[-2.697493 ,  2.5639   ],
       [-2.6228921,  2.5515208]], dtype=float32), label_ids=None, metrics={'test_runtime': 0.2053, 'test_samples_per_second': 9.74, 'test_steps_per_second': 4.87})

To see how your model fared you can compare it to the [GLUE Benchmark leaderboard](https://gluebenchmark.com/leaderboard).

## Hyperparameter search

The `Trainer` supports hyperparameter search using [optuna](https://optuna.org/) or [Ray Tune](https://docs.ray.io/en/latest/tune/). For this last section you will need either of those libraries installed, just uncomment the line you want on the next cell and run it.

In [29]:
! pip install optuna
! pip install ray[tune]

During hyperparameter search, the `Trainer` will run several trainings, so it needs to have the model defined via a function (so it can be reinitialized at each new run) instead of just having it passed. We jsut use the same function as before:

In [30]:
def model_init() -> Any:
    return AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

And we can instantiate our `Trainer` like before:

In [31]:
trainer = Trainer(
    model_init=model_init,
    args=args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset[validation_key],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


The method we call this time is `hyperparameter_search`. Note that it can take a long time to run on the full dataset for some of the tasks. You can try to find some good hyperparameter on a portion of the training dataset by replacing the `train_dataset` line above by:
```python
train_dataset = encoded_dataset["train"].shard(index=1, num_shards=10) 
```
for 1/10th of the dataset. Then you can run a full training on the best hyperparameters picked by the search.

In [32]:
best_run = trainer.hyperparameter_search(n_trials=10, direction="maximize")
assert isinstance(best_run, BestRun)

[I 2026-08-21 16:15:25,043] A new study created in memory with name: no-name-20aabb02-f927-4c44-b72c-8c43d2ec128a


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory w

Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.566341,0.562119,0.000000
2,0.512090,0.570040,0.376242
3,0.456146,0.587642,0.400709
4,0.466401,0.615478,0.392117
5,0.460885,0.602519,0.413749


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-08-21 17:41:52,584] Trial 9 finished with value: 0.4936257221427369 and parameters: {'learning_rate': 2.5251020501648355e-05, 'num_train_epochs': 4, 'seed': 17, 'per_device_train_batch_size': 8}. Best is trial 1 with value: 0.5300972900950771.


The `hyperparameter_search` method returns a `BestRun` objects, which contains the value of the objective maximized (by default the sum of all metrics) and the hyperparameters it used for that run.

In [33]:
best_run

BestRun(run_id='1', objective=0.5300972900950771, hyperparameters={'learning_rate': 8.721678842948556e-05, 'num_train_epochs': 3, 'seed': 28, 'per_device_train_batch_size': 64}, run_summary=None)

You can customize the objective to maximize by passing along a `compute_objective` function to the `hyperparameter_search` method, and you can customize the search space by passing a `hp_space` argument to `hyperparameter_search`. See this [forum post](https://discuss.huggingface.co/t/using-hyperparameter-search-in-trainer/785/10) for some examples.

To reproduce the best training, just set the hyperparameters in your `TrainingArgument` before creating a `Trainer`:

In [34]:
for n, v in best_run.hyperparameters.items():
    setattr(trainer.args, n, v)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory w

Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.609818,0.669957,0.000000
2,0.558336,0.650107,0.154314
3,0.445929,0.643660,0.267643


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3207, training_loss=0.5425681627759467, metrics={'train_runtime': 621.2677, 'train_samples_per_second': 41.291, 'train_steps_per_second': 5.162, 'total_flos': 120777926900772.0, 'train_loss': 0.5425681627759467, 'epoch': 3.0})

Don't forget to [upload your model](https://huggingface.co/transformers/model_sharing.html) on the [Model Hub](https://huggingface.co/models). You can then use it only to generate results like the one shown in the first picture of this notebook!